In [2]:
library(readxl)
library(readr)
library(dplyr)
library(ggplot2)
library(tidyr)
library(ggrepel)
library(networkD3)
library(tibble)
library(ggalluvial)
library(RColorBrewer)
library(scales)
library(viridis)

In [3]:
immunoglobulin <- read.table("/vol/projects/CIIM/cohorts_old/500FG/Molecular_phenotype/500FG_log2_immunoglobulin_levels.txt", header = TRUE, sep = "", stringsAsFactors = FALSE)
head(immunoglobulin)
dim(immunoglobulin)
any(is.na(immunoglobulin))
sum(is.na(immunoglobulin))

,IgG,IgA,IgM,IgG1,IgG2,IgG3,IgG4,Sum.IgG
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
HV001,2.835924,1.5945485,0.8318772,2.150560,0.8797058,-2.321928,-2.1844246,2.744161
HV002,3.249445,1.4168397,-1.2863042,2.438293,1.0840643,-1.599462,-2.2515388,3.014355
HV003,3.364572,0.3334237,0.3448285,2.786596,1.0214797,-1.434403,-2.0588937,3.253989
HV004,3.392317,0.1634987,0.9411063,2.655352,1.4750849,-1.434403,-0.8889687,3.320485
HV005,3.510962,-0.1681228,1.4956952,2.427606,2.1667154,-1.434403,-1.2863042,3.412782
HV006,3.584963,1.2016339,1.4541759,2.769772,2.0908534,-1.089267,-0.3770696,3.622930


[1] 489   8

[1] FALSE

[1] 0

In [5]:
basicPhenos = read.csv("/vol/projects/yzhang/500FG_aging/output/02_Olink/Age_group_basicPhenos.csv")
rownames(basicPhenos) = basicPhenos$ID_500fg

## Filter a feature matrix and phenotype table to shared sample IDs and align rows in the same order
prep_filter_match <- function(feature_mat, basicPhenos) {
  idx <- rownames(feature_mat)[rownames(feature_mat) %in% rownames(basicPhenos)]  # keep feature_mat order
  feature_filter <- feature_mat[idx, , drop = FALSE]
  pheno_match <- basicPhenos[rownames(feature_filter), , drop = FALSE]
  list(feature_filter = feature_filter, basicPhenos_filter_match = pheno_match)
}
res <- prep_filter_match(immunoglobulin, basicPhenos)
immunoglobulin_filter <- res$feature_filter
basicPhenos_filter_match <- res$basicPhenos_filter_match
##one missing gender value
basicPhenos_filter_match$Gender <- trimws(basicPhenos_filter_match$Gender)
basicPhenos_filter_match$Gender[basicPhenos_filter_match$Gender == ""] <- NA

head(immunoglobulin_filter)
dim(immunoglobulin_filter)
head(basicPhenos_filter_match)
dim(basicPhenos_filter_match)

,IgG,IgA,IgM,IgG1,IgG2,IgG3,IgG4,Sum.IgG
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
HV001,2.835924,1.5945485,0.8318772,2.150560,0.8797058,-2.321928,-2.1844246,2.744161
HV002,3.249445,1.4168397,-1.2863042,2.438293,1.0840643,-1.599462,-2.2515388,3.014355
HV003,3.364572,0.3334237,0.3448285,2.786596,1.0214797,-1.434403,-2.0588937,3.253989
HV004,3.392317,0.1634987,0.9411063,2.655352,1.4750849,-1.434403,-0.8889687,3.320485
HV005,3.510962,-0.1681228,1.4956952,2.427606,2.1667154,-1.434403,-1.2863042,3.412782
HV006,3.584963,1.2016339,1.4541759,2.769772,2.0908534,-1.089267,-0.3770696,3.622930


[1] 482   8

,X,ID_500fg,Gender,Age,Height,Weight,BMI,LivingSituation,Education,Ethnicity,FarmAnimal,Pets,Travel,DayInMonth,monthinYear,dayInYear,year,AgeGroup
,<int>,<chr>,<chr>,<int>,<int>,<int>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<chr>
HV001,304,HV001,male,27,178,70,22.09317,Married,University degree,European,Once a month,Yes,No,24,7,204,2013,Group 5
HV002,305,HV002,male,26,178,68,21.46194,Cohabitation,University degree,European,Rarely,No,Yes,24,7,204,2013,Group 4
HV003,257,HV003,female,24,173,68,22.72044,Married,University degree,European,Rarely,Yes,Yes,26,8,236,2013,Group 4
HV004,258,HV004,female,20,173,65,21.71807,Married,University degree,European,Rarely,Yes,No,26,8,236,2013,Group 2
HV005,254,HV005,female,22,178,64,20.19947,Married,University student,European,Once a month,Yes,Yes,5,9,245,2013,Group 3
HV006,530,HV006,female,24,168,58,20.54989,Married,University degree,European,Rarely,Yes,No,NA,NA,NA,NA,Group 4


[1] 482  18

In [6]:
####model 2: age + immunoglobulinusing FDR age not scale + gender
all_results <- list()

for (i in seq_len(ncol(immunoglobulin_filter))) {

  # Build a minimal modeling table
  data <- data.frame(
    feature =immunoglobulin_filter[, i],
    age = basicPhenos_filter_match$Age,
    Gender = factor(basicPhenos_filter_match$Gender)  # convert chr to factor
  )

  # Linear regression
  mod <- lm(feature ~ age + Gender, data = data)

  # Extract coefficient and p-value for age (not Gender)
  cf <- summary(mod)$coefficients
  res <- cf["age", c("Estimate", "Pr(>|t|)")]

  # Store results
  result <- data.frame(
    feature  = colnames(immunoglobulin_filter)[i],
    estimate = res["Estimate"],
    p        = res["Pr(>|t|)"]
  )

  # Signed -log10(p)
  result$value <- -log10(result$p) * result$estimate

  all_results[[i]] <- result
}

final_result <- do.call(rbind, all_results)
final_result$padj <- p.adjust(final_result$p, method = "BH")
final_result$sig <- ifelse(final_result$padj < 0.05, "sig", "no")
head(final_result)
dim(final_result)
print(table(final_result$sig))
write.csv(final_result, file = "/vol/projects/yzhang/500FG_aging/output/08_immunoglobulin/immunoglobulin_age_gender_FDR.csv")

,feature,estimate,p,value,padj,sig
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
Estimate,IgG,-0.0008073224,0.4688979843,-0.0002655458,0.5358834106,no
Estimate1,IgA,0.0108772125,0.0001071556,0.0431823702,0.0008572451,sig
Estimate2,IgM,-0.0053371194,0.0352829504,-0.0077518196,0.0705659008,no
Estimate3,IgG1,-0.0028358536,0.0174610334,-0.0049852323,0.0465627556,sig
Estimate4,IgG2,0.0061802909,0.0008521803,0.0189702083,0.0034087213,sig
Estimate5,IgG3,-0.0035759784,0.1547176510,-0.0028981880,0.2475482415,no


[1] 8 6


 no sig 
  5   3 


In [7]:
save.image("/vol/projects/yzhang/500FG_aging/output/08_immunoglobulin/immunoglobulin_age_gender_FDR.Rdata")

In [3]:
load("/vol/projects/yzhang/500FG_aging/output/08_immunoglobulin/immunoglobulin_age_gender_FDR.Rdata")

In [4]:
final_result_sig <- final_result %>%
  filter(sig == "sig") %>%
  select(feature, estimate, p, padj)
head(final_result_sig)
dim(final_result_sig)
write_excel_csv(final_result_sig,"/vol/projects/yzhang/500FG_aging/output/08_immunoglobulin/immunoglobulin_age_gender_FDR_sig.csv")

,feature,estimate,p,padj
,<chr>,<dbl>,<dbl>,<dbl>
Estimate1,IgA,0.010877212,0.0001071556,0.0008572451
Estimate3,IgG1,-0.002835854,0.0174610334,0.0465627556
Estimate4,IgG2,0.006180291,0.0008521803,0.0034087213


[1] 3 4